In [1]:
!pip install supabase
!pip install boto3 botocore

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 112.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 295.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 307.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 206.2 MB/s  0:00:00
  Attempting uninstall: pydantic-core━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/17 [zstandard]
    Found existing installation: pydantic_core 2.27.2━━━━━━━━━  2/17 [zstandard]
    Uninstalling pydantic_core-2.27.2:━━━━━━━━━━━━━━━━━━━━━━━━  2/17 [zstandard]
      Successfully uninstalled pydantic_core-2.27.2━━━━━━━━━━━  2/17 [zstandard]
  Attempting uninstall: pydanticm╺━━━━━━━━━━━━━━━━━━━━━━━  7/17 [strictyaml]
    Found existing installation: pydantic 2.10.6━━━━━━━━━━━━━━  7/17 [strictyaml]
    Uninstalling pydantic-2.10.6:m━━━━━━━━━━━━━━━━━━━━━━━  7/17 [strictyaml]
      Successfully uninstalled pydantic-2.10.6━━━━━━━━━━━━━━━━  7/17 [strictyaml]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 

In [2]:
import pandas as pd
from supabase import create_client, Client
import os
import zipfile
import os
import boto3
import botocore
from datetime import datetime, timezone

In [3]:
SAVED_DIR = "saved"
def parse_folder_datetime(folder_name: str) -> datetime | None:
    try:
        iso_like = folder_name.replace("T", " ").replace("Z", "")
        date_part, time_part = iso_like.split(" ")

        h, m, s, ms = time_part.split("-")
        iso_string = f"{date_part}T{h}:{m}:{s}.{ms}Z"

        return datetime.strptime(
            iso_string, "%Y-%m-%dT%H:%M:%S.%fZ"
        ).replace(tzinfo=timezone.utc)

    except Exception:
        return None


def find_latest_folder() -> str:
    candidates = []

    for name in os.listdir(SAVED_DIR):
        path = os.path.join(SAVED_DIR, name)

        if not os.path.isdir(path):
            continue

        if not name.startswith("2"):
            continue

        dt = parse_folder_datetime(name)
        if dt:
            candidates.append((dt, name))

    return max(candidates, key=lambda x: x[0])[1]

In [4]:
url = "https://gchiqqahgionsvsehtip.supabase.co"
key = os.getenv("key")
supabase: Client = create_client(url, key)
date = find_latest_folder()

In [5]:
# insert cpe changes into database
df = pd.read_csv(f'saved/{date}/changes_cpe.csv')
df.columns = ['cve_id', 'action', 'cpe', 'date_changed']
records = df.to_dict(orient="records")
response = supabase.table("CPE_Changes").insert(records).execute()

In [6]:
# insert cwe changes into database
df = pd.read_csv(f'saved/{date}/changes_cwe.csv')
df.columns = ['cve_id', 'action', 'cwe', 'date_changed']
records = df.to_dict(orient="records")
response = supabase.table("CWE_Changes").insert(records).execute()

In [ ]:
df = pd.read_csv(f'saved/{date}/cve_reformatted.csv')

cwe_disc = ["CWE-119", "CWE-20", "CWE-200", "CWE-269", "CWE-287",
            "CWE-311", "CWE-330", "CWE-345", "CWE-400", "CWE-610",
            "CWE-662", "CWE-665", "CWE-668", "CWE-682", "CWE-697",
            "CWE-74", "CWE-755", "CWE-834"]
df_cwe_cate = pd.read_csv("NERC-FixV2W/preprocess/csv_file/2021aug/cwe_category.csv")
df_cwe_view = pd.read_csv("NERC-FixV2W/preprocess/csv_file/2021aug/cwe_view.csv")
df_cwe_cate.fillna('*', inplace=True)
df_cwe_view.fillna('*', inplace=True)
cwe_prohibited = df_cwe_cate["ID"].tolist() + df_cwe_view["ID"].tolist()
cwe_proh_set = set(cwe_prohibited)

df = df.fillna('*')

cwe_disc_set = set(cwe_disc)


def check_discouraged(cwe_string):
    if cwe_string == '*' or pd.isna(cwe_string):
        return False
    return any(c.strip() in cwe_disc_set for c in cwe_string.split(';'))


def check_prohibited(cwe_string):
    if cwe_string == '*' or pd.isna(cwe_string):
        return False
    return any(c.strip() in cwe_proh_set for c in cwe_string.split(';'))


df['is_discouraged'] = df['MatchingCWE'].apply(check_discouraged)
df['is_prohibited'] = df['MatchingCWE'].apply(check_prohibited)

df.columns = ['cve_id', 'last_modified', 'description', 'matching_cwe',
              'keywords', 'matching_cpe', 'is_discouraged', 'is_prohibited']

records = df.to_dict(orient="records")
chunk_size = 5000

for i in range(0, len(records), chunk_size):
    chunk = records[i:i + chunk_size]
    response = (
        supabase
        .table("CVE_metadata")
        .upsert(chunk, on_conflict="cve_id")
        .execute()
    )
    print(f"Upserted rows {i} to {i + len(chunk) - 1}")

In [10]:
# add .pt for future reference
aws_access_key_id = os.environ.get('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')
endpoint_url = os.environ.get('AWS_S3_ENDPOINT')
region_name = os.environ.get('AWS_DEFAULT_REGION')
bucket_name = os.environ.get('AWS_S3_BUCKET')

if not all([aws_access_key_id, aws_secret_access_key, endpoint_url, region_name, bucket_name]):
    raise ValueError("One or more connection variables are empty. Please check your S3 connection.")

session = boto3.session.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key
)

s3_resource = session.resource(
    's3',
    config=botocore.client.Config(signature_version='s3v4'),
    endpoint_url=endpoint_url,
    region_name=region_name
)

bucket = s3_resource.Bucket(bucket_name)


def upload_file_to_s3(local_file_path, s3_key):
    print(f"Uploading {local_file_path} -> {s3_key}")
    bucket.upload_file(local_file_path, s3_key)


def upload_directory_to_s3(local_directory, s3_prefix):
    num_files = 0
    for root, dirs, files in os.walk(local_directory):
        for filename in files:
            file_path = os.path.join(root, filename)
            relative_path = os.path.relpath(file_path, local_directory)
            s3_key = os.path.join(s3_prefix, relative_path).replace("\\", "/")  # handle Windows paths
            upload_file_to_s3(file_path, s3_key)
            num_files += 1
    return num_files


def list_objects(prefix):
    for obj in bucket.objects.filter(Prefix=prefix):
        print(obj.key)


s3_base_prefix = f"models/{date}"

model_file_path = f"saved/{date}/transe_model_nll.pt"
if not os.path.isfile(model_file_path):
    raise ValueError(f"Model file not found at {model_file_path}")

upload_file_to_s3(model_file_path, f"{s3_base_prefix}/model.pt")

id_conversions_dir = f"saved/{date}/id_conversions"
if os.path.isdir(id_conversions_dir):
    num_id_files = upload_directory_to_s3(id_conversions_dir, f"{s3_base_prefix}/id_conversions")
    print(f"Uploaded {num_id_files} id conversion files")
else:
    print(f"No id_conversions folder found at {id_conversions_dir}, skipping")

print("\nS3 contents in folder:")
list_objects(s3_base_prefix)

Uploading saved/2026-04-06T20-32-44-867Z/transe_model_nll.pt -> models/2026-04-06T20-32-44-867Z/model.pt
Uploading saved/2026-04-06T20-32-44-867Z/id_conversions/entity2id.pkl -> models/2026-04-06T20-32-44-867Z/id_conversions/entity2id.pkl
Uploading saved/2026-04-06T20-32-44-867Z/id_conversions/triples_id.npy -> models/2026-04-06T20-32-44-867Z/id_conversions/triples_id.npy
Uploading saved/2026-04-06T20-32-44-867Z/id_conversions/id2entity.pkl -> models/2026-04-06T20-32-44-867Z/id_conversions/id2entity.pkl
Uploading saved/2026-04-06T20-32-44-867Z/id_conversions/relation2id.pkl -> models/2026-04-06T20-32-44-867Z/id_conversions/relation2id.pkl
Uploading saved/2026-04-06T20-32-44-867Z/id_conversions/id2relation.pkl -> models/2026-04-06T20-32-44-867Z/id_conversions/id2relation.pkl
Uploaded 5 id conversion files

S3 contents in folder:
models/2026-04-06T20-32-44-867Z/id_conversions/entity2id.pkl
models/2026-04-06T20-32-44-867Z/id_conversions/id2entity.pkl
models/2026-04-06T20-32-44-867Z/id_con